# EDA

In [4]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect(database=':memory:')

query = """
    SELECT 
        d.Doc_ID,
        CASE WHEN a.Author_Type = 'AI' THEN 1 ELSE 0 END AS Target_Is_AI,
        COUNT(t.Chunk_ID) AS Feature_Sentence_Count,
        ROUND(AVG(LENGTH(t.Raw_Text)), 2) AS Feature_Avg_Sentence_Length,
        ROUND(STDDEV(LENGTH(t.Raw_Text)), 2) AS Feature_Burstiness,
        ROUND(MAX(LENGTH(t.Raw_Text)), 2) AS Feature_Max_Sentence_Length,
        ROUND(MIN(LENGTH(t.Raw_Text)), 2) AS Feature_Min_Sentence_Length
    FROM 'data_organizer/FINAL_1_authors.parquet' a
    JOIN 'data_organizer/FINAL_3_documents.parquet' d ON a.Author_ID = d.Author_ID
    JOIN 'data_organizer/FINAL_4_text_chunks.parquet' t ON d.Doc_ID = t.Doc_ID
    GROUP BY 
        d.Doc_ID, 
        a.Author_Type
    -- Filter out documents with too few sentences to calculate a valid standard deviation
    HAVING COUNT(t.Chunk_ID) > 2; 
"""

ml_df = con.execute(query).df()
ml_df.fillna(0, inplace=True) 

display(ml_df.head(20))

,Doc_ID,Target_Is_AI,Feature_Sentence_Count,Feature_Avg_Sentence_Length,Feature_Burstiness,Feature_Max_Sentence_Length,Feature_Min_Sentence_Length
0,4963b9f6-4384-4e98-9b2b-51b7a7529e5e,1,17,109.88,24.40,164,76
1,5fa6a3fe-7c34-411d-9c3a-0d6535c8a0cd,1,17,140.00,45.71,260,69
2,1529d851-d956-41a7-a300-63db0f82596b,1,27,100.33,20.80,148,56
3,fd516f26-a3a6-487e-be71-e9eadd38fac0,1,24,129.67,27.63,180,71
4,ed7a602f-23df-4768-bab7-7a04c97b732a,1,15,120.13,30.77,171,62
5,588438b1-8949-46b1-b435-0655b7be6b19,1,13,126.77,40.77,200,61
6,15f9955d-9b04-42c2-9a0b-3ba66818e741,1,17,116.47,29.96,170,60
7,350556ae-ea90-4dcd-b3f0-d6c5e475d256,1,20,144.95,47.02,242,63
8,b0ac73f6-423e-4d49-bf8c-8d32cd6ab5b0,1,15,133.60,26.74,165,74
9,9a70f814-8a51-43a7-a78e-fab13e412245,1,16,138.31,50.05,236,54


In [5]:
import pandas as pd

ml_df['Author_Label'] = ml_df['Target_Is_AI'].map({1: 'AI', 0: 'Human'})

total_rows = len(ml_df)
print(f"Total documents in feature matrix: {total_rows:,}\n")

distribution = ml_df['Author_Label'].value_counts()
print("--- Class Distribution ---")
print(distribution.to_string())
print()

averages = ml_df.groupby('Author_Label')[['Feature_Burstiness', 'Feature_Avg_Sentence_Length']].mean().round(2)
print("--- Average Metrics by Author Type ---")
print(averages.to_string())

Total documents in feature matrix: 96,007

--- Class Distribution ---
Author_Label
AI       51967
Human    44040

--- Average Metrics by Author Type ---
              Feature_Burstiness  Feature_Avg_Sentence_Length
Author_Label                                                 
AI                         60.84                       151.94
Human                      58.46                       134.53
